# 02 — Dummy baseline (Người 1)


## Mở project
**Colab:** đặt đầy đủ thư mục project tại `MyDrive/student-dropout-prediction`,
gồm notebook, `common/`, `requirements.txt` và `data/raw/data.csv`.
Nếu thư mục có tên khác, sửa `PROJECT_ROOT` ở cell dưới. Mở mỗi notebook rồi chạy từ trên xuống.
**VS Code/Jupyter:** mở notebook từ thư mục `notebooks/`; cell tự tìm thư mục gốc.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/student-dropout-prediction')
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [here, *here.parents]
                         if (p / 'common' / 'data_io.ipynb').is_file()), here)

if not (PROJECT_ROOT / 'common' / 'data_io.ipynb').is_file():
    raise FileNotFoundError('Sửa PROJECT_ROOT tới thư mục chứa common/ và requirements.txt.')
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)


## Thư viện
Colab không cần `venv`, nhưng vẫn cần đúng thư viện. Cell này đọc `requirements.txt`.
Nếu Colab yêu cầu khởi động lại runtime sau khi cài, thực hiện rồi chạy lại từ đầu.


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
import json
import pandas as pd
from IPython.display import display
%run common/data_io.ipynb
%run common/splits.ipynb

DATA_PATH = PROJECT_ROOT / 'data/raw/data.csv'
SPLIT_PATH = PROJECT_ROOT / 'data/splits/splits.json'
ARTIFACTS = PROJECT_ROOT / 'artifacts'
TABLES = PROJECT_ROOT / 'reports/tables'
FIGURES = PROJECT_ROOT / 'reports/figures'
for folder in [SPLIT_PATH.parent, ARTIFACTS, TABLES, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Đặt CSV gốc tại {DATA_PATH}')


Chạy notebook 01 trước để tạo bộ chia chung. Dummy, Logistic và RF trong bản này dùng CPU.
Logistic/RF tìm tham số qua 5 fold Train; pipeline fit tiền xử lý riêng mỗi fold.
Lựa chọn mô hình dựa trên Validation; không đọc Test ở bước này.


In [ ]:
%run common/training.ipynb
model, metrics, run_dir = run_validation('dummy', DATA_PATH, SPLIT_PATH, ARTIFACTS)
print('Lượt chạy:', run_dir.relative_to(PROJECT_ROOT))
display(pd.Series({k: metrics[k] for k in ['accuracy', 'macro_f1', 'dropout_f2']}))


Mỗi lần chạy tạo thư mục mới chứa model, metrics, dự đoán, cấu hình và phiên bản thư viện. Bàn giao cả thư mục lượt chạy cho người tích hợp.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np
report_figures = FIGURES / run_dir.parent.name / run_dir.name
report_tables = TABLES / run_dir.parent.name / run_dir.name
report_figures.mkdir(parents=True, exist_ok=True)
report_tables.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{
    'model': run_dir.parent.name, 'run_id': run_dir.name,
    **{k: metrics[k] for k in ['accuracy', 'macro_f1', 'dropout_f2']},
}]).to_csv(report_tables / 'validation_metrics.csv', index=False)
ConfusionMatrixDisplay(np.asarray(metrics['confusion_matrix']), display_labels=LABELS).plot(cmap='Blues')
plt.tight_layout()
plt.savefig(report_figures / 'confusion_matrix.png', dpi=160)
plt.show()
